In [1]:
!nvidia-smi

Thu Nov 12 10:24:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # The maximum length of our SQL + Question. 2048 is plenty.
dtype = None # Auto-detect (will be float16 on T4)
load_in_4bit = True # This is the QLoRA part

# Load the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 10:32:41.335000 356 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 10:32:41.377000 356 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded successfully!


In [5]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank of the LoRA matrices (higher = more parameters, more capacity)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth recommends 0 for speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves memory
    random_state = 3407,
)

Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [6]:
from datasets import load_dataset

# 1. Load the dataset from Hugging Face
dataset = load_dataset("b-mc2/sql-create-context", split="train")

print(f"Total examples: {len(dataset)}")
print("\n--- Example 1 ---")
print(f"Schema: {dataset[0]['context']}")
print(f"Question: {dataset[0]['question']}")
print(f"SQL: {dataset[0]['answer']}")

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Total examples: 78577

--- Example 1 ---
Schema: CREATE TABLE head (age INTEGER)
Question: How many heads of the departments are older than 56 ?
SQL: SELECT COUNT(*) FROM head WHERE age > 56


In [7]:
# This is the template we will inject every row into
prompt_template = """<|im_start|>system
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and a context. You must output the SQL query that answers the question.
<|im_end|>
<|im_start|>user
### Context:
{context}

### Question:
{question}
<|im_end|>
<|im_start|>assistant
{answer}<|im_end|>"""

def format_prompts(examples):
    # Get the lists of contexts, questions, and answers
    contexts = examples["context"]
    questions = examples["question"]
    answers = examples["answer"]

    texts = []
    for context, question, answer in zip(contexts, questions, answers):
        # Fill in the template
        text = prompt_template.format(context=context, question=question, answer=answer)
        texts.append(text)

    return { "text": texts }

# Apply this formatting to the whole dataset
dataset = dataset.map(format_prompts, batched=True)

print("Formatted example:")
print(dataset[0]["text"])

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

Formatted example:
<|im_start|>system
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and a context. You must output the SQL query that answers the question.
<|im_end|>
<|im_start|>user
### Context:
CREATE TABLE head (age INTEGER)

### Question:
How many heads of the departments are older than 56 ?
<|im_end|>
<|im_start|>assistant
SELECT COUNT(*) FROM head WHERE age > 56<|im_end|>


In [8]:
# Split the dataset into 95% for training and 5% for testing
split_dataset = dataset.train_test_split(test_size=0.05, seed=3407)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(f"Training examples: {len(train_dataset)}")
print(f"Testing examples: {len(test_dataset)}")

Training examples: 74648
Testing examples: 3929


In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Take a subset of 3000 examples for fast training
train_subset = train_dataset.select(range(3000))

# 2. Set up the training arguments
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_subset,
    dataset_text_field = "text", # The column containing our formatted prompt
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # How many examples to process at once
        gradient_accumulation_steps = 4, # Accumulate gradients to simulate a larger batch size
        warmup_steps = 5, # Gently ramp up the learning rate
        max_steps = 60,   # How many training steps to run. 60 steps is about 1 epoch on 3000 examples.
        learning_rate = 2e-4, # How fast the model learns
        fp16 = not is_bfloat16_supported(), # Use FP16 on T4
        bf16 = is_bfloat16_supported(),
        logging_steps = 10, # Print the loss every 10 steps
        optim = "adamw_8bit", # Memory-efficient optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Don't send logs to Weights & Biases
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
10,2.303148
20,1.000987
30,0.724757
40,0.658442
50,0.605388
60,0.586548


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Training complete!


In [10]:
# 1. Save the LoRA adapter locally in Colab
model.save_pretrained("qwen2.5-coder-7b-text-to-sql-lora")
tokenizer.save_pretrained("qwen2.5-coder-7b-text-to-sql-lora")
print("Model and tokenizer saved locally!")

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-coder-7b-text-to-sql-lora/tokenizer_config.json.


Model and tokenizer saved locally!


In [11]:
from unsloth import FastLanguageModel

# 1. Switch the model to inference mode (turns off training dropouts)
FastLanguageModel.for_inference(model)

# 2. A brand new prompt the model has NEVER seen
test_prompt = """<|im_start|>system
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and a context. You must output the SQL query that answers the question.
<|im_end|>
<|im_start|>user
### Context:
CREATE TABLE employees (id INT, name VARCHAR, department VARCHAR, salary INT, hire_date DATE)

### Question:
What are the names of the employees in the 'Engineering' department?
<|im_end|>
<|im_start|>assistant
"""

# 3. Tokenize the prompt
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# 4. Ask the model to generate the answer
outputs = model.generate(
    **inputs,
    max_new_tokens=100, # Max length of the SQL query
    use_cache=True,
    temperature=0.1, # Low temperature means less creative, more deterministic
)

# 5. Decode the output and print it
response = tokenizer.batch_decode(outputs)[0]

# 6. Extract just the SQL part
sql_query = response.split("<|im_start|>assistant")[1].replace("<|im_end|>", "").strip()
print("\n--- GENERATED SQL ---")
print(sql_query)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- GENERATED SQL ---
SELECT name FROM employees WHERE department = "Engineering"


In [12]:
from unsloth import FastLanguageModel

# 1. Load the ORIGINAL base model (without your fine-tuning)
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(base_model)

# 2. Use the same prompt
inputs = base_tokenizer([test_prompt], return_tensors="pt").to("cuda")

# 3. Generate the answer
outputs = base_model.generate(
    **inputs,
    max_new_tokens=100,
    use_cache=True,
    temperature=0.1,
)

# 4. Decode and print
response = base_tokenizer.batch_decode(outputs)[0]
base_sql = response.split("<|im_start|>assistant")[1].replace("<|im_end|>", "").strip()

print("\n--- BASE MODEL (WITHOUT YOUR TRAINING) ---")
print(base_sql)

==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BASE MODEL (WITHOUT YOUR TRAINING) ---
```sql
SELECT name FROM employees WHERE department = 'Engineering';
```


In [ ]:
from huggingface_hub import login

# Paste your token inside the quotes
login(token="**YOUR_HF_TOKEN**") # CHANGE THIS

In [15]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "crazyintrovert7/qwen2.5-coder-7b-text-to-sql-lora"

# Create the repository
api.create_repo(repo_id=repo_id, exist_ok=True)
print(f"Repository created: https://huggingface.co/{repo_id}")

Repository created: https://huggingface.co/crazyintrovert7/qwen2.5-coder-7b-text-to-sql-lora


In [16]:
# Push the model and tokenizer to the hub
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Successfully pushed to Hugging Face!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 49.4kB /  162MB            

Validating                    :           |   0%            

Saved model to https://huggingface.co/crazyintrovert7/qwen2.5-coder-7b-text-to-sql-lora


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

... (2 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpjzh2hu8w/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Successfully pushed to Hugging Face!
